In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path

plt.style.use('seaborn-v0_8')
%matplotlib inline

PROJECT_ROOT = Path('/home/praruj/stress-detection')
DATA_PATH    = PROJECT_ROOT / 'WESAD'

print("All imports OK")

Window size: (42000,)samples(60s)
Step Size : 21000 samples (30s)
Subjects:16


In [ ]:
subject = 'S2'
pkl_path = DATA_PATH / subject / f'{subject}.pkl'

with open(pkl_path, 'rb') as f:
    data = pickle.load(f, encoding='latin1')

print("Top-level keys  :", list(data.keys()))
print("Signal sources  :", list(data['signal'].keys()))
print("Chest signals   :", list(data['signal']['chest'].keys()))
print("Wrist signals   :", list(data['signal']['wrist'].keys()))
print("Label shape     :", data['label'].shape)
print("Subject ID      :", data['subject'])

signals shape: (4255300, 4)
label shape :(4255300,)
signal columns:['ECG', 'EDA', 'EMG', 'Temp']


In [ ]:
labels    = data['label']
label_map = {0:'Undefined', 1:'Baseline', 2:'Stress', 
             3:'Amusement', 4:'Meditation'}

print(f"Label distribution for Subject {subject}:\n")
for val, name in label_map.items():
    count = np.sum(labels == val)
    pct   = count / len(labels) * 100
    bar   = '█' * int(pct / 2)
    print(f"  {name:<12} (label {val}): {count:>9,} samples  {pct:5.1f}%  {bar}")

In [ ]:
changes    = np.where(np.diff(labels) != 0)[0] + 1
boundaries = [0] + list(changes) + [len(labels)]

label_names_full = {0:'Undefined', 1:'Baseline', 2:'Stress',
                    3:'Amusement', 4:'Meditation', 
                    6:'Unknown-6', 7:'Unknown-7'}
FS = 700

print(f"{'#':<3} {'Lbl':<4} {'Condition':<12} {'Start':>10} {'End':>10} {'Duration':>10}")
print("-" * 60)

for i in range(len(boundaries)-1):
    start  = boundaries[i]
    end    = boundaries[i+1]
    lbl    = labels[start]
    dur_s  = (end - start) / FS
    usable = "✓" if lbl in [1,2,3,4] else "✗"
    print(f"{i:<3} {lbl:<4} {label_names_full.get(lbl,'?'):<12} "
          f"{start:>10,} {end:>10,} {dur_s:>8.1f}s  {usable}")

total_usable = sum(
    (boundaries[i+1] - boundaries[i]) / FS
    for i in range(len(boundaries)-1)
    if labels[boundaries[i]] in [1,2,3,4]
)
print(f"\nTotal usable data: {total_usable:.1f}s ({total_usable/60:.1f} min)")

In [ ]:
print("=" * 60)
print("CHEST SENSORS (700 Hz)")
print("=" * 60)
for sig_name, sig_data in data['signal']['chest'].items():
    arr = sig_data.flatten()
    print(f"  {sig_name:<6}  shape: {str(arr.shape):<14}  "
          f"min: {arr.min():8.3f}  max: {arr.max():8.3f}  "
          f"mean: {arr.mean():8.3f}")

print("\n" + "=" * 60)
print("WRIST SENSORS (mixed rates)")
print("=" * 60)
for sig_name, sig_data in data['signal']['wrist'].items():
    arr = np.array(sig_data).flatten()
    print(f"  {sig_name:<6}  shape: {str(arr.shape):<14}  "
          f"min: {arr.min():8.3f}  max: {arr.max():8.3f}  "
          f"mean: {arr.mean():8.3f}")

In [ ]:
ecg    = data['signal']['chest']['ECG'].flatten()
labels = data['label']

fig, axes = plt.subplots(3, 1, figsize=(14, 8))
conditions = [
    (1, 'Baseline',  'steelblue'),
    (2, 'Stress',    'crimson'),
    (3, 'Amusement', 'seagreen'),
]

for ax, (label_val, name, color) in zip(axes, conditions):
    indices = np.where(labels == label_val)[0]
    mid     = len(indices) // 2
    segment = ecg[indices[mid : mid + 7000]]  # 10 seconds
    
    ax.plot(segment, color=color, linewidth=0.6)
    ax.set_title(f'{name}  —  10 seconds of ECG', fontsize=11)
    ax.set_ylabel('mV')
    ax.set_xlabel('Sample index (700 Hz)')
    ax.grid(True, alpha=0.3)

plt.suptitle(f'Subject {subject} — ECG Signal by Condition', 
             fontsize=13, fontweight='bold')
plt.tight_layout()

out_dir = PROJECT_ROOT / 'outputs'
out_dir.mkdir(exist_ok=True)
plt.savefig(out_dir / f'{subject}_ecg_by_condition.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to outputs/")

In [ ]:
eda = data['signal']['chest']['EDA'].flatten()

fig, axes = plt.subplots(3, 1, figsize=(14, 8))

for ax, (label_val, name, color) in zip(axes, conditions):
    indices = np.where(labels == label_val)[0]
    # Show full condition block (downsampled for speed)
    segment = eda[indices[::70]]  # every 70th sample = 10 Hz display
    time_s  = np.arange(len(segment)) / 10
    
    ax.plot(time_s, segment, color=color, linewidth=1.0)
    ax.set_title(f'{name}  —  Full EDA signal', fontsize=11)
    ax.set_ylabel('μS')
    ax.set_xlabel('Time (seconds)')
    ax.grid(True, alpha=0.3)

plt.suptitle(f'Subject {subject} — Electrodermal Activity by Condition',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(out_dir / f'{subject}_eda_by_condition.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to outputs/")

In [ ]:
SIGNALS_TO_USE = ['ECG', 'EDA', 'EMG', 'Temp']
signal_colors  = ['steelblue', 'darkorange', 'purple', 'firebrick']

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

# Show 60 seconds of Baseline
baseline_idx = np.where(labels == 1)[0]
start        = baseline_idx[0]
end          = start + 60 * FS  # 60 seconds

for ax, sig_name, color in zip(axes, SIGNALS_TO_USE, signal_colors):
    sig_data = data['signal']['chest'][sig_name].flatten()
    segment  = sig_data[start:end]
    time_s   = np.arange(len(segment)) / FS
    
    ax.plot(time_s, segment, color=color, linewidth=0.7)
    ax.set_ylabel(sig_name, fontsize=10)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Time (seconds)')
plt.suptitle(f'Subject {subject} — All Chest Signals (60s Baseline)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(out_dir / f'{subject}_all_signals_baseline.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to outputs/")